<a href="https://colab.research.google.com/github/aldo02032004/social-media-kol-analysis/blob/main/mention_cleaning_media_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mention cleaning and media classification

1. **Load** a monitoring export from a Google Sheets / Drive link.
2. **Clean**: strip news boilerplate, URLs, hashtags and @mentions; drop emoji-only posts, broadcast spam (same text more than 5 times) and duplicates.
3. **Split** rows into social media and news/online, and take the 5 with the most followers from each.
4. **Top issues**: Gemini groups them into 5 social-media and 5 news issues. Each gets a sentiment score, the key opinion leader (social account with the most followers) and the public figures involved.

Needs a Gemini API key in Colab **Secrets** as `GOOGLE_API_KEY` (step 4 only).

## 1. Install

In [ ]:
%pip install -q "great[text] @ git+https://github.com/azmkto/GREAT-Tools.git" google-genai

## 2. Load data

In [2]:
import html, re
from urllib.error import HTTPError

import pandas as pd
from IPython.display import display

from great.text import normalize_slang

SOURCE = "https://docs.google.com/spreadsheets/d/1Q4-Uknh0Dl_9KpqkvZpCIZYwlolsU63a/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
COLS = ["No", "Headline", "Mentions", "Date", "Link", "Media", "Sentiment", "Author", "Followers"]


def xlsx_url(link):
    """Link Google Sheets/Drive -> link download .xlsx."""
    if m := re.search(r"docs\.google\.com/spreadsheets/d/([\w-]+)", link):
        return f"https://docs.google.com/spreadsheets/d/{m[1]}/export?format=xlsx"
    if m := re.search(r"drive\.google\.com/file/d/([\w-]+)", link):
        return f"https://drive.google.com/uc?export=download&id={m[1]}"
    return link


for attempt in range(3):   # Google kadang membalas 400 sesaat
    try:
        sheets = pd.read_excel(xlsx_url(SOURCE), sheet_name=None, skiprows=1)
        break
    except HTTPError:
        if attempt == 2:
            raise
    except ValueError as e:
        raise ValueError("Gagal baca file. Pastikan aksesnya 'Anyone with the link'.") from e
for d in sheets.values():
    d.columns = d.columns.astype(str).str.strip()
df_raw = max(sheets.values(), key=lambda d: len(set(d.columns) & set(COLS)))   # sheet yang paling mirip export
if missing := [c for c in COLS if c not in df_raw]:
    print("Kolom hilang:", missing)
df_raw = df_raw.dropna(subset=["Mentions"]).reset_index(drop=True)[[c for c in COLS if c in df_raw]]
print(f"{len(df_raw)} baris")
display(df_raw.head())

8747 baris


,No,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers
0,1,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:59:55,https://twitter.com/web/statuses/2098094128915...,Twitter,Negative,@lifecapture_,7
1,2,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:59:51,https://twitter.com/web/statuses/2098094113166...,Twitter,Negative,@jstcallmeaul,41
2,3,NaN,"RT PRABOWO DIHUJAT SE ASEAN, laporan yang masu...",2026-09-10 23:58:52,https://twitter.com/web/statuses/2098093865895...,Twitter,Negative,@takayatamiskin,2
3,4,Tiorita: Pemkab Langkat Siap Kolaborasi Dukung...,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...",2026-09-10 23:57:56,https://bbsnews.id/2026/09/10/tiorita-pemkab-l...,News,Positive,bbsnews.id,0
4,5,NaN,"RT Belum, belum over. War is over kalo prabowo...",2026-09-10 23:56:36,https://twitter.com/web/statuses/2098093293938...,Twitter,Negative,@jstcallmeaul,41


## 3. Clean and filter

In [3]:
BOILERPLATE = re.compile("|".join([
    r"Dilarang keras mengambil konten.*?ANTARA\.",
    r"Kirim\s*Komentar menjadi tanggung-?jawab Anda sesuai UU ITE\.",
    r"Yuk Subscribe.*?(?=\n\n|\Z)",
    r"Follow (WA Channel|our Official).*?(?=\n\n|\Z)",
    r"Like our Official Facebook.*?(?=\n\n|\Z)",
    r"Uploader\s*:\s*\S+",
    r"Tanggal Tayang\s*:.*",
    r"Selengkapnya baca di\s*:\s*\S+",
    r"\bRT\b",
]), flags=re.IGNORECASE | re.DOTALL)
NOISE = re.compile(r"https?://\S+|www\.\S+|[#@]\S+")   # url, hashtag, @akun
EMOJI = re.compile("[\U0001F300-\U0001FAFF\U0001F1E6-\U0001F1FF\U00002700-\U000027BF\U00002600-\U000026FF\U00002190-\U000021FF]+")
MAX_REPEAT = 5   # teks yang sama lebih dari ini = broadcast/bot


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = NOISE.sub(" ", BOILERPLATE.sub(" ", html.unescape(text)))
    return re.sub(r"\n{2,}", "\n", re.sub(r"[ \t]+", " ", text)).strip()


def is_low_quality(text):
    """Isinya cuma emoji/simbol: kurang dari 3 kata atau 10 karakter."""
    words = re.sub(r"[^\w\s]", " ", EMOJI.sub(" ", text)).split()
    return len(words) < 3 or len(" ".join(words)) < 10


df = df_raw.assign(clean_text=df_raw["Mentions"].map(clean_text))
df = df[df["clean_text"].str.len() > 20]
n = len(df)
df = df[~df["clean_text"].map(is_low_quality)].reset_index(drop=True)
print(f"Emoji/simbol saja: {n} -> {len(df)} baris")
df["doc_id"] = [f"DOC-{i:05d}" for i in range(len(df))]

key = df["clean_text"].str.strip().str.lower()
n = len(df)
df = df[key.map(key.value_counts()) <= MAX_REPEAT]   # semua kemunculan teks spam dibuang
print(f"Spam (>{MAX_REPEAT}x): {n} -> {len(df)} baris")

dup = df["Headline"].astype(str).str.strip().str.lower() + "||" + df["clean_text"].str[:200].str.strip().str.lower()
n = len(df)
df = df[~dup.duplicated()].reset_index(drop=True)
print(f"Duplikat: {n} -> {len(df)} baris")

df["search_text"] = (df["Headline"].str.lower().fillna("") + " " + df["clean_text"]).str.lower().map(normalize_slang)
df["clean_text"] = df["clean_text"].str.lower()
display(df[["doc_id", "Mentions", "clean_text", "search_text", "Link", "Sentiment", "Author"]].head())

Emoji/simbol saja: 8288 -> 8278 baris
Spam (>5x): 8278 -> 5211 baris
Duplikat: 5211 -> 2138 baris


,doc_id,Mentions,clean_text,search_text,Link,Sentiment,Author
0,DOC-00003,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...","foto bersama : pelaksana tugas bupati langkat,...",tiorita: pemkab langkat siap kolaborasi dukung...,https://bbsnews.id/2026/09/10/tiorita-pemkab-l...,Positive,bbsnews.id
1,DOC-00007,Sebanyak 11 kapal gabungan telah diterjunkan u...,sebanyak 11 kapal gabungan telah diterjunkan u...,prabowo perintahkan tni al optimalkan pencaria...,https://news.republika.co.id/berita/tl5qxq393/...,Neutral,news.republika.co.id
2,DOC-00011,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,"prabowo mantap""penyerahan bantuan beca listrik...",prabowo bagus&quot;penyerahan bantuan beca lis...,https://www.youtube.com/watch?v=BXMdYvVoOuA,Positive,SAHABAT KANG DEDI MULYADI69
3,DOC-00012,@RA_leather ketika dirugikan atas perbuatan me...,ketika dirugikan atas perbuatan melawan hukum ...,ketika dirugikan atas perbuatan melawan hukum...,https://twitter.com/web/statuses/2098092101775...,Negative,@mochmasykur12
4,DOC-00014,Crop shirt linen Wowo ❤️ #music #fashiontrends...,crop shirt linen wowo ❤️,crop shirt linen prabowo ❤️ #music #fashiontre...,https://www.youtube.com/watch?v=DqQsnPE8Dl0,Positive,Life wth Pv


## 4. Social media vs news

In [4]:
SOCIAL = ("instagram", "twitter", "x.com", "threads", "youtube", "tiktok", "facebook")


def media_category(media):
    if not isinstance(media, str) or not media.strip():
        return "Tidak Diketahui"
    return "Media Sosial" if any(p in media.lower() for p in SOCIAL) else "Berita/Online"


df["Media_Category"] = df["Media"].map(media_category)
df["Followers"] = pd.to_numeric(df["Followers"], errors="coerce")
print(df["Media_Category"].value_counts().to_string())

# 5 per kategori dengan followers terbanyak (proxy jangkauan)
selected = pd.concat([df[df["Media_Category"] == c].sort_values("Followers", ascending=False).head(5)
                      for c in ("Berita/Online", "Media Sosial")], ignore_index=True)
display(selected[["No", "Media_Category", "Media", "Headline", "Mentions", "Sentiment", "Author", "Followers"]])

Media_Category
Berita/Online    1146
Media Sosial      992


,No,Media_Category,Media,Headline,Mentions,Sentiment,Author,Followers
0,4,Berita/Online,News,Tiorita: Pemkab Langkat Siap Kolaborasi Dukung...,"FOTO BERSAMA : Pelaksana Tugas Bupati Langkat,...",Positive,bbsnews.id,0
1,9,Berita/Online,News,Prabowo Perintahkan TNI AL Optimalkan Pencaria...,Sebanyak 11 kapal gabungan telah diterjunkan u...,Neutral,news.republika.co.id,0
2,18,Berita/Online,News,Puan Maharani Desak Pencarian Maksimal! 5 Jurn...,PUAN MAHARANI: Ketua DPR RI Puan Maharani mend...,Positive,sinfonews.com,0
3,23,Berita/Online,News,Becak Listrik Bantuan Prabowo Mulai Dijajal di...,Becak listrik bantuan Presiden Prabowo Subiant...,Neutral,jabar.tribunnews.com,0
4,30,Berita/Online,News,"Amien Rais Singgung Prabowo: Pidatonya Galak, ...",RakyatPos.id &ndash; Ketua Dewan Pertimbangan ...,Neutral,www.rakyatpos.id,0
5,14,Media Sosial,Twitter,NaN,@RA_leather ketika dirugikan atas perbuatan me...,Negative,@mochmasykur12,51
6,13,Media Sosial,Youtube,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,PRABOWO MANTAP&quot;PENYERAHAN BANTUAN BECA LI...,Positive,SAHABAT KANG DEDI MULYADI69,0
7,16,Media Sosial,Youtube,Crop shirt linen Wowo ❤️ #music #fashiontrends...,Crop shirt linen Wowo ❤️ #music #fashiontrends...,Positive,Life wth Pv,0
8,25,Media Sosial,Youtube,"Usai Karhutla, Pohon Sawit Muncul, Prabowo Min...","Usai Karhutla, Pohon Sawit Muncul, Prabowo Min...",Neutral,Trijaya FM,0
9,27,Media Sosial,Youtube,Tingkah Gemas Orangutan Digandeng Wapres Gibra...,Tingkah Gemas Orangutan Digandeng Wapres Gibra...,Neutral,Roll Channel Solo,0


## 5. Top issues (Gemini)

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel, Field

N = 5   # isu per kategori
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"),   # limit/server sibuk dicoba ulang otomatis
                      http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=5)))


class Isu(BaseModel):
    isu_utama: str
    ringkasan: str = Field(description="maksimal 3 kalimat")
    sumber_no: list[int] = Field(description="nilai 'No' dari data yang dibahas")
    tokoh: str = Field(description="tokoh publik/politik yang paling disorot, pisahkan dengan koma; "
                                   "'Tidak ada tokoh spesifik' kalau tidak ada")


class Hasil(BaseModel):
    media_sosial: list[Isu] = Field(min_length=N, max_length=N)
    berita_online: list[Isu] = Field(min_length=N, max_length=N)


PERSONA = """You're Syahganda Nainggolan — seorang intelektual-aktivis Indonesia, mantan aktivis ITB era 80-an
yang di-DO karena menentang Orde Baru, meraih gelar Doktor di bidang perburuhan dari FISIP UI,
dan dikenal sebagai tokoh oposisi yang konsisten mengkritik kekuasaan yang sedang berkuasa.
Kamu adalah salah satu petinggi KAMI (Koalisi Aksi Menyelamatkan Indonesia), pendiri
Sabang-Merauke Circle (SMC), dan punya concern besar pada isu buruh, kesejahteraan sosial,
dan nasionalisme.

Cara berpikirmu:
- Kritis dan skeptis terhadap narasi resmi pemerintah/elite penguasa.
- Tajam membedah ketimpangan sosial, nasib buruh/rakyat kecil, dan ancaman terhadap nasionalisme.
- Lugas, berani, dan konfrontatif.
- Mencari pola dan motif di balik data, bukan sekadar mendeskripsikan permukaan teks."""

data = "\n\n---\n\n".join(
    f"[No: {r.No} | Kategori: {r.Media_Category} | Media: {r.Media}]\nHeadline: {r.Headline}\nMentions: {r.Mentions}"
    for r in selected.fillna("-").itertuples())
prompt = f"""{PERSONA}

Berikut {N} item Berita/Online dan {N} item Media Sosial, masing-masing dengan 'No'.
Buat {N} isu utama media sosial dan {N} isu utama berita online yang dianalisis secara kritis.
Kalau isu yang benar-benar berbeda kurang dari {N}, pecah sudut pandang/sub-isu dari data yang sama.

{data}"""

resp = client.models.generate_content(
    model="gemini-flash-latest", contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Hasil))
if resp.parsed is None:
    raise RuntimeError(f"Jawaban Gemini tidak sesuai format, jalankan ulang cell ini.\n{resp.text[:500]}")


def sentiment(nos):
    """Skor = porsi positif dari post positif+negatif; selisih <= 20% = kontroversial."""
    s = selected.loc[selected["No"].isin(nos), "Sentiment"].astype(str).str.strip().str.lower()
    pos, neg = s.isin(["positif", "positive", "pos"]).sum(), s.isin(["negatif", "negative", "neg"]).sum()
    if pos + neg == 0:
        return "Tidak Diketahui"
    if abs(pos - neg) / (pos + neg) <= 0.2:
        label = "Kontroversial"
    else:
        label = "Positif" if pos > neg else "Negatif"
    return f"{round(pos / (pos + neg), 2)} ({label})"


def key_opinion_leader(nos):
    """Akun media sosial dengan followers terbanyak di antara sumber isu."""
    social = selected[selected["No"].isin(nos) & (selected["Media_Category"] == "Media Sosial")]
    if social.empty:
        return "Tidak ada (isu didominasi sumber berita)"
    top = social.sort_values("Followers", ascending=False).iloc[0]
    if pd.isna(top["Followers"]):
        return f"{top['Author']} (followers tidak diketahui)"
    return f"{top['Author']} ({top['Followers']:,.0f} followers)".replace(",", ".")


def issue_table(issues):
    return pd.DataFrame([{"no": i, "nama_isu": x.isu_utama, "sentimen": sentiment(x.sumber_no),
                          "key_opinion_leader": key_opinion_leader(x.sumber_no), "summary": x.ringkasan,
                          "politically_exposed_person": x.tokoh} for i, x in enumerate(issues, 1)])


pd.set_option("display.max_colwidth", None)
print("=== Isu media sosial ===")
display(issue_table(resp.parsed.media_sosial))
print("=== Isu berita online ===")
display(issue_table(resp.parsed.berita_online))